# nb03b · Backbone training run

Same architecture as nb03 (d_model=768, n_layers=8, n_heads=12) but with the price encoder backbone **unfrozen**.

Key differences from nb03:
- `freeze_backbone=False` — the 8-layer transformer trains instead of staying at random-init frozen weights. In nb03, `freeze_backbone=True` meant only the projection head and LoRA adapters learned; the backbone was random throughout 10k steps.
- `lr_encoder=5e-6` — 4× lower than the 2e-5 that caused monotonic val degradation in temp.txt. Backbone needs a much smaller LR than the projection head or the encoder drifts faster than the head can adapt.
- `RESUME=False` — nb03's checkpoint used a frozen random backbone, so it's not a useful warm start here; training must begin from scratch.
- Checkpoint saved to `nb03b_best.pt` (separate from nb03).

Data and library code live in `src/` pulled fresh from GitHub via the setup cell.

## 0 · Environment — mount Drive, pull latest `src/` from git, put it on the path

In [ ]:
import os, sys

IN_COLAB = 'google.colab' in sys.modules or 'COLAB_GPU' in os.environ
IN_VSCODE = 'VSCODE_PID' in os.environ or 'VSCODE_CWD' in os.environ
if IN_VSCODE:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    REPO = '/content/JEPA-quant'
    if not os.path.exists(REPO):
        !git clone https://github.com/shreyasnat2804/JEPA-quant.git {REPO}
    else:
        !git -C {REPO} pull

    SRC      = f'{REPO}/src'
    DATA_DIR = '/content/drive/MyDrive/Colab Notebooks/JEPA-QUANT/data/raw/stocks'
    CKPT_DIR = '/content/drive/MyDrive/Colab Notebooks/JEPA-QUANT/checkpoints'
else:
    REPO     = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..'))
    SRC      = os.path.join(REPO, 'src')
    DATA_DIR = os.path.join(REPO, 'data', 'raw', 'stocks')
    CKPT_DIR = os.path.join(REPO, 'checkpoints')

if SRC not in sys.path:
    sys.path.insert(0, SRC)

os.makedirs(CKPT_DIR, exist_ok=True)
print(f'IN_COLAB={IN_COLAB}  SRC={SRC}')
print(f'DATA_DIR={DATA_DIR}')
print(f'CKPT_DIR={CKPT_DIR}')

In [ ]:
# Verify library is up-to-date (Colab only)
if IN_COLAB:
    import subprocess
    result = subprocess.run(['git', '-C', REPO, 'log', '-1', '--format=%h %ci %s'], capture_output=True, text=True)
    print('HEAD:', result.stdout.strip())
    behind = subprocess.run(['git', '-C', REPO, 'rev-list', '--count', 'HEAD..origin/main'], capture_output=True, text=True)
    n = behind.stdout.strip()
    if n != '0':
        print(f'WARNING: {n} commit(s) behind origin/main — re-run setup cell')
    else:
        print('Up-to-date with origin/main')

In [ ]:
# Autoreload — picks up edits to jepa_quant/*.py without kernel restart.
# jepa_quant.config is excluded: frozen dataclasses can't reassign __class__ on reload.
try:
    import imp  # noqa: F401
except ModuleNotFoundError:
    import importlib, types
    _imp = types.ModuleType('imp')
    _imp.reload = importlib.reload  # type: ignore[attr-defined]
    import sys as _sys
    _sys.modules['imp'] = _imp

%load_ext autoreload
%autoreload 2
%aimport -jepa_quant.config

## 1 · Dependencies

In [ ]:
# Install runtime deps (transformer backend only — no uni2ts/Moirai).
# torchao is uninstalled to prevent PEFT's dispatch from crashing on Colab's
# preinstalled torchao 0.10.0 (needs >=0.16.0 but we don't use quantization).
if IN_COLAB:
    %pip install -q transformers peft accelerate einops matplotlib pyarrow
    %pip uninstall -q -y torchao
    import numpy as np
    print('numpy', np.__version__)

## 2 · Hugging Face login

In [ ]:
# HF login is required when USE_FOUNDATION_MODELS=True (LM predictor downloads Qwen).
# With USE_FOUNDATION_MODELS=False (transformer predictor) this cell is a no-op.
USE_FOUNDATION_MODELS = True  # set False to skip HF download and LM predictor

if USE_FOUNDATION_MODELS:
    try:
        if IN_COLAB:
            from google.colab import userdata
            HF_TOKEN = userdata.get('HF_TOKEN')
        else:
            HF_TOKEN = os.environ.get('HF_TOKEN', '')
        from huggingface_hub import login
        login(token=HF_TOKEN, add_to_git_credential=False)
        print('HF login OK')
    except Exception as e:
        print(f'HF login skipped: {e}')

## 3 · Configuration

Same encoder size as nb03: **d_model=768, n_layers=8, n_heads=12**.

**`freeze_backbone=False`** — backbone trains from random init at `lr_encoder=5e-6`.  
**`RESUME=False`** — always train from scratch (nb03 checkpoint has a frozen random backbone, not a useful warm start).

In [ ]:
import dataclasses as dc
import torch
from jepa_quant import JEPAConfig
from jepa_quant.config import (
    PriceEncoderConfig, PredictorConfig, DataConfig, TrainConfig
)

REGULARIZER = 'vicreg'  # 'vicreg' or 'codebook'
CONTEXT_LENGTH = 64
HORIZON        = 16
LATENT_DIM     = 256

# nb03b always trains from scratch — nb03's checkpoint used a frozen random backbone.
RESUME = False

_lr_proj    = 1e-4
_lr_adapter = 5e-5
_warmup     = 500

cfg = JEPAConfig(
    price_encoder=PriceEncoderConfig(
        d_model=768,
        n_layers=8,
        n_heads=12,
        context_length=CONTEXT_LENGTH,
        latent_dim=LATENT_DIM,
        freeze_backbone=False,   # train the backbone (key diff from nb03)
    ),
    predictor=PredictorConfig(
        backend='lm' if USE_FOUNDATION_MODELS else 'transformer',
        latent_dim=LATENT_DIM,
    ),
    data=DataConfig(
        data_dir=DATA_DIR,
        context_length=CONTEXT_LENGTH,
        horizon=HORIZON,
    ),
    train=TrainConfig(
        max_steps=10000,
        warmup_proj_steps=_warmup,
        val_every=500,
        es_patience=6,
        es_min_delta=0.002,
        batch_size=256,
        lr_proj=_lr_proj,
        lr_adapter=_lr_adapter,
        lr_encoder=5e-6,         # 4× lower than temp.txt's 2e-5 that caused drift
    ),
    regularizer=REGULARIZER,
)

print(cfg)
print(f'\nlr_proj={_lr_proj:.0e}  lr_adapter={_lr_adapter:.0e}  lr_encoder=5e-6')
print(f'device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"}')

## 4 · Data

In [ ]:
from jepa_quant.data.price_dataset import build_dataloaders

train_loader, val_loader = build_dataloaders(cfg)
print(f'Train batches: {len(train_loader)}  Val batches: {len(val_loader)}')
sample = next(iter(train_loader))
print(f'context: {sample["context"].shape}  target: {sample["target"].shape}')

## 5 · Build components & train

- Best `val_jepa` checkpoint saved to `nb03b_best.pt` on Drive after every improvement.
- Early stopping fires after 6 consecutive non-improving val checks (Δ < 0.002).
- Backbone (`freeze_backbone=False`) trains at `lr_encoder=5e-6` alongside projection head and LoRA adapters.

In [ ]:
import torch
from jepa_quant.training.trainer import build_components, JEPATrainer

CKPT_PATH = os.path.join(CKPT_DIR, 'nb03b_best.pt')

components = build_components(cfg)

def save_best(ckpt: dict) -> None:
    torch.save(ckpt, CKPT_PATH)
    print(f'  [ckpt] New best val_jepa={ckpt["val_jepa"]:.4f} at step {ckpt["step"]} → {CKPT_PATH}')

trainer = JEPATrainer(cfg, components, train_loader, val_loader, on_new_best=save_best)
history = trainer.train()

## 6 · Diagnostics

In [ ]:
import matplotlib.pyplot as plt

steps        = [r['step'] for r in history]
jepa_loss    = [r['jepa'] for r in history]
total_loss   = [r['loss'] for r in history]
val_steps    = [r['step'] for r in history if 'val_jepa' in r]
val_jepa     = [r['val_jepa'] for r in history if 'val_jepa' in r]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(steps, total_loss, label='total loss', alpha=0.7)
axes[0].plot(steps, jepa_loss,  label='jepa loss',  alpha=0.7)
axes[0].set_title('Training loss')
axes[0].legend()
axes[0].set_xlabel('step')

if val_steps:
    axes[1].plot(val_steps, val_jepa, marker='o', label='val_jepa')
    axes[1].set_title('Validation JEPA loss')
    axes[1].set_xlabel('step')
    axes[1].legend()

# Regularization metrics (VICReg or codebook)
reg_keys = [k for k in history[0] if k.startswith('reg/')]
if reg_keys:
    for k in reg_keys:
        axes[2].plot(steps, [r.get(k, float('nan')) for r in history], label=k.split('/')[-1], alpha=0.8)
    axes[2].set_title('Regularization')
    axes[2].legend()
    axes[2].set_xlabel('step')
else:
    axes[2].set_visible(False)

plt.tight_layout()
plt.show()

print(f'Final step: {steps[-1]}  best val_jepa: {trainer._best_val_jepa:.4f}')
print(f'Checkpoint path: {CKPT_PATH}')